# Explore the record yourself

The [walkthrough](walkthrough.ipynb) was generated from one file, the run's
*record*. This notebook opens that file and finds the same things directly, to
show what the record holds and how to reach it. In practice you will mostly ask
the front end (Claude Code) about a run and it reads the record for you, with
`./run.sh status`, `show` and `explore`; this is what it reads.

To run it yourself, open it from this folder in a Jupyter kernel using the
repository's environment (`uv sync --group docs`). The outputs below were
produced when the documentation was built.

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[3]  # docs/site/examples/ctd_blind -> the repo
sys.path.insert(0, str(REPO))

from octomod.result.node import Node
from octomod.result.runstore import RunStore

store = RunStore(Path("record/ctd_blind_local_v3_opus55_20261004_022720.h5"))
root = Node(store)  # the record, addressed by key
root.children()

Every part of the run has a **key**. What was asked, and how it ended:

In [ ]:
print((root / "intent" / "question").get())
(root / "outcome").get()

The **ledger** is the run's diary: every module, check, agent call and decision,
in order, with timestamps.

In [ ]:
import pandas as pd

ledger = pd.DataFrame(store.ledger_rows())
ledger["event"].value_counts().head(10)

Every agent call, with its round, model, cost and duration:

In [ ]:
calls = ledger[ledger["event"] == "agent_call"]
calls[["round", "module", "role", "model_resolved", "prompt_bytes", "cost_usd", "duration_s"]]

Every answer at a gate, and the note that came with it:

In [ ]:
ledger[ledger["event"] == "human_gate"][["round", "module", "approved", "note"]]

Each agent's **prompt** and validated **reply** sit under its round's
`agent_calls`. Here, the start of the prompt the expansion agent received, and
one of the candidates it proposed:

In [ ]:
call = root / "rounds" / "round_0" / "agent_calls" / "expansion"
print(call.child("prompt").text()[:1200])

In [ ]:
candidate = call.child("reply").get()["candidates"][2]
{k: candidate[k] for k in ("id", "label", "adds_parameters")}

The fit of every candidate in both rounds, and each round's model selection:

In [ ]:
rows = []
for rnd in ("round_0", "round_1"):
    cands = root / "rounds" / rnd / "candidates"
    for cid in cands.children():
        est = cands.child(cid).child("estimation").get()
        x = est["best_x_unscaled"]
        rows.append({"round": rnd, "candidate": cid, "-log L": est["best_fval"],
                     **{p: x.get(p) for p in ("kmod", "cp", "nb", "kon_I")}})
pd.DataFrame(rows).set_index(["round", "candidate"]).round(4)

In [ ]:
sel = (root / "rounds" / "round_1" / "selection").get()
pd.DataFrame(sel["values"]).T.assign(weight=pd.Series(sel["weights"])).round(3)

Per data point, the record keeps the measured value beside each candidate's
prediction at its fit — enough to redraw any fit without simulating:

In [ ]:
import re

import matplotlib.pyplot as plt

groups = (root / "rounds" / "round_1" / "attribution" / "per_candidate" / sel["winner"] / "groups").get()
points = []
for key, g in groups.items():
    m = re.match(r"obs_phos(\d)__sc_ctd_ms_t(\d+)__", key)
    points.append((int(m.group(2)), int(m.group(1)), g["measured"][0], g["simulated"][0]))
df = pd.DataFrame(points, columns=["time", "phosphates", "measured", "simulated"])
fig, axes = plt.subplots(1, 4, figsize=(11, 2.5))
for ax, (t, d) in zip(axes, df.sort_values("phosphates").groupby("time")):
    ax.bar(d["phosphates"], d["measured"], color="#ddd", label="measured")
    ax.plot(d["phosphates"], d["simulated"], "o-", color="#2ca02c", label=sel["winner"])
    ax.set_title(f"t = {t} min")
axes[0].legend(fontsize=7);

The same file also loads as one **typed object**, validated against the
schema (`docs/RESULT-SCHEMA.md`), for code that wants attributes rather than
keys:

In [ ]:
run = store.load()
run.outcome.winner, [run.round(k).selection.winner for k in (0, 1)]

From a terminal, the same record reads with the engine's CLI (the run file
must sit in `runs/`):

```bash
./run.sh status  --run-id ctd_blind_local_v3_opus55_20261004_022720
./run.sh show    --run-id ctd_blind_local_v3_opus55_20261004_022720 rounds/round_1/decisions_md
./run.sh explore --run-id ctd_blind_local_v3_opus55_20261004_022720     # ls, cd, cat, gates, ledger …
```